# ── 0. 配置区 ──────────────────────────────────────────────

In [ ]:
# ── 0. 配置区 ──────────────────────────────────────────────
from pathlib import Path

# 要处理的 FiftyOne dataset 名称列表
datasets_list = [
    "sahi_null_v2_ms1_0605-0621_40_ok",
    "sahi_null_v2_ms1_0710-0726_36_ok",
    "sahi_null_v2_ms1_0726-0809_11_ok",
    "sahi_null_v2_ms1_0809-0823_34_ok",
    "sahi_null_v2_ms2_0726-0809_13_ok",
    "sahi_null_v2_ms2_0809-0823_10_ok",
    "sahi_null_v2_sw1_0605-0613_07_ok",
]

# Step 1：在 FiftyOne 中运行检测评估时使用的参数
GT_FIELD   = "ground_truth"
PRED_FIELDS = [
    "small_slices_a03_yolo11n_custom7null_cv1_ms2_0809-0823_10_ok_8",
    "small_slices_a04_yolo11n_custom7null_cv1_ms2_0809-0823_10_ok_8",
    "small_slices_a03_yolo11s_custom7null_cv1_ms2_0809-0823_10_ok_4",
    "small_slices_a03_yolo11s_custom7null_cv1_ms2_0809-0823_10_ok_16",
    "small_slices_a03_yolo11s_custom7null_cv1_ms2_0809-0823_10_ok_8",
    "small_slices_a04_yolo11s_custom7null_cv1_ms2_0809-0823_10_ok_16",
    "small_slices_a04_yolo11n_custom7null_cv1_ms2_0809-0823_10_ok_4",
    "small_slices_a04_yolo11s_custom7null_cv1_ms2_0809-0823_10_ok_8",
]
IOU            = 0.5
METHOD         = "coco"
CLASSWISE      = True
COMPUTE_MAP    = True
confidence_thresholds = [0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 0.91, 0.92]

# Step 2：导出路径
out_dir = Path(".")
out_dir.mkdir(parents=True, exist_ok=True)

# ── 1. 初始化日志 ──────────────────────────────────────────

In [ ]:
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")

# ── 2. 查看所有 Datasets ──────────────────────────────────

In [ ]:
# ── Step 0：列出所有 datasets，确认名称拼写 ─────────────────
import fiftyone as fo

logger.info("Step 0 开始：列出所有 datasets")
datasets = fo.list_datasets()
for dataset_name in datasets:
    print(f'"{dataset_name}",')
logger.info(f"Step 0 完成：共 {len(datasets)} 个 dataset")

# ── 3. Step 1：在 FiftyOne 中对每个 dataset 进行多阈值评估 ─

In [ ]:
# ── Step 1：对 datasets_list 中每个 dataset 的各 pred_field 进行多阈值评估 ──
# 输入：datasets_list, PRED_FIELDS, confidence_thresholds
# 输出：FiftyOne datasets 中写入 eval_key 结果

import fiftyone as fo
from fiftyone import ViewField as F

logger.info("Step 1 开始：多 dataset × 多 pred_field × 多阈值评估")


def safe_tag(name: str, max_len: int = 40) -> str:
    s = name.replace("/", "_").replace(" ", "_").replace("-", "_")
    return s[:max_len]


for dataset_name in datasets_list:
    logger.info(f"开始处理 dataset: {dataset_name}")
    try:
        dataset = fo.load_dataset(dataset_name)
    except Exception as e:
        logger.error(f"加载 dataset 失败 {dataset_name}: {e}")
        continue

    schema_keys = set(dataset.get_field_schema().keys())

    if GT_FIELD not in schema_keys:
        logger.warning(f"GT_FIELD '{GT_FIELD}' 不存在于 {dataset_name}，跳过")
        continue

    for pred_field in PRED_FIELDS:
        if pred_field not in schema_keys:
            logger.warning(f"pred_field '{pred_field}' 不存在于 {dataset_name}，跳过")
            continue

        pred_tag = safe_tag(pred_field)

        for conf_thresh in confidence_thresholds:
            eval_key = f"fresh_{pred_tag}_conf_{conf_thresh*100:02.0f}"
            try:
                view = dataset.filter_labels(
                    pred_field, F("confidence") > conf_thresh, only_matches=True
                )
                results = view.evaluate_detections(
                    pred_field, gt_field=GT_FIELD, eval_key=eval_key,
                    method=METHOD, iou=IOU, classwise=CLASSWISE, compute_mAP=COMPUTE_MAP,
                )
                m = results.metrics()
                logger.info(
                    f"  conf>{conf_thresh:.2f} | P={m.get('precision', 0):.3f} "
                    f"R={m.get('recall', 0):.3f}"
                )
            except Exception as e:
                logger.error(f"评估失败 {eval_key}: {e}")

logger.info("Step 1 完成：多阈值评估结束")

# ── 4. Step 2：提取评估结果导出到 Excel ──────────────────

In [ ]:
# ── Step 2：从每个 dataset 读取 eval_key 结果，汇总到 Excel ──
# 输入：datasets_list 中各 dataset 的 evaluation keys
# 输出：每个 dataset 对应的 evaluation_{dataset_name}.xlsx

import pandas as pd
from ty_eval_tools import get_evaluation_data_like_gui

logger.info("Step 2 开始：提取评估结果并导出 Excel")

for dataset_name in datasets_list:
    logger.info(f"处理 dataset: {dataset_name}")
    try:
        dataset = fo.load_dataset(dataset_name)
    except Exception as e:
        logger.error(f"加载 dataset 失败 {dataset_name}: {e}")
        continue

    try:
        eval_keys = dataset.list_evaluations()
    except Exception as e:
        logger.error(f"获取 eval_keys 失败 {dataset_name}: {e}")
        continue

    all_data = []
    for eval_key in eval_keys:
        try:
            metrics, info = get_evaluation_data_like_gui(dataset, eval_key)
            row = {
                "eval_key": eval_key,
                "eval_type": info.config.type,
                "eval_method": info.config.method,
                "average_confidence": metrics.get("average_confidence", 0),
                "support": metrics.get("support", 0),
                "accuracy": metrics.get("accuracy", 0),
                "iou_threshold": getattr(info.config, "iou", 0.5),
                "precision": metrics.get("precision", 0),
                "recall": metrics.get("recall", 0),
                "f1_score": metrics.get("fscore", 0),
                "map": metrics.get("map"),
                "mar": metrics.get("mar"),
                "true_positives": metrics.get("tp", 0),
                "false_positives": metrics.get("fp", 0),
                "false_negatives": metrics.get("fn", 0),
            }
            all_data.append(row)
        except Exception as e:
            logger.error(f"处理 eval_key={eval_key} 失败: {e}")

    out_xlsx = out_dir / f"evaluation_{dataset_name}.xlsx"
    try:
        df = pd.DataFrame(all_data)
        df.to_excel(str(out_xlsx), index=False)
        logger.info(f"已保存: {out_xlsx} ({len(df)} rows)")
    except Exception as e:
        logger.error(f"保存 Excel 失败 {out_xlsx}: {e}")

logger.info("Step 2 完成：所有 dataset 评估结果已导出")

# ── 5. 验证：抽查单个 dataset 结果 ──────────────────────

In [ ]:
# ── 验证：展示第一个 dataset 的评估结果 ────────────────────
import pandas as pd

sample_name = datasets_list[0]
sample_xlsx = out_dir / f"evaluation_{sample_name}.xlsx"

try:
    df_check = pd.read_excel(str(sample_xlsx))
    logger.info(f"验证：{sample_xlsx}, shape={df_check.shape}")
    display(df_check.head(10))
except Exception as e:
    logger.error(f"读取验证文件失败: {e}")